In [ ]:
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA

def statement_to_dict_df(file, statement_name):
    # Extract the ticker from the filename by removing the statement name and file extension
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    df = pd.read_csv(file)

    metric_col = df.columns[0]

    rows = []

    # Iterate over each date column (starting from the second column) and create a dictionary of metrics for that date
    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows)

In [3]:
from pathlib import Path

DATA_DIR = Path("../data")
BALANCE_SHEET_DIR = DATA_DIR / "balance_sheet"
CASH_FLOW_DIR = DATA_DIR / "cashflows"
FINANCIALS_DIR = DATA_DIR / "financials"

tickers = set()

for file in BALANCE_SHEET_DIR.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)

print(f"Found {len(tickers)} companies")

Found 92 companies


In [4]:
master_rows = []

for ticker in tickers:

    try:
        bs_file = BALANCE_SHEET_DIR / f"{ticker}_balancesheet.csv"
        cf_file = CASH_FLOW_DIR / f"{ticker}_cashflow.csv"
        fin_file = FINANCIALS_DIR / f"{ticker}_financials.csv"

        bs = statement_to_dict_df(bs_file, "balancesheet")
        cf = statement_to_dict_df(cf_file, "cashflow")
        fin = statement_to_dict_df(fin_file, "financials")

        company_df = (
            bs.merge(cf, on=["company", "date"], how="outer")
              .merge(fin, on=["company", "date"], how="outer")
        )

        master_rows.append(company_df)

        print(f"Processed {ticker}")

    except Exception as e:
        # companies that are missing one of the statements will be skipped
        print(f"Failed {ticker}: {e}")


Processed SNAP
Processed MVIS
Processed ASML
Processed CHGG
Failed ARCIZZX: 'company'
Processed CARS
Processed WDAY
Processed PTON
Processed EXPE
Processed L360.F
Failed SKYA-USD: 'company'
Processed EBAY
Failed ACKOYXX: 'company'
Failed NAYAX: 'company'
Processed SHOP
Failed DLC36116-USD: 'company'
Failed BHARATAGRI.BO: 'company'
Failed 5126.T: 'company'
Failed CDCETH-USD: 'company'
Processed ADSK
Processed CTEV
Processed RIVN
Processed ORCL
Processed IAC
Failed EPIGZZX: 'company'
Processed AMAT
Processed TEAM
Processed GEMI
Processed AVGO
Processed COIN
Failed ALTRZZX: 'company'
Processed 0M5.MU
Failed DIGGX: 'company'
Processed VSCO
Processed AMZN
Processed MBLY
Processed PAYC
Processed SNPS
Failed TYPTF: 'company'
Processed HPE
Failed HNS-USD: 'company'
Processed 600223.SS
Processed FRSH
Processed PLTK
Processed FORM
Processed PAYO
Failed SOND.SW: 'company'
Processed GLOO
Processed ERIC
Processed UPWK
Processed CRM
Processed 002058.SZ
Processed ETOR
Processed SONO
Failed MMF21370-U

In [ ]:
dataset = pd.concat(master_rows, ignore_index=True)

dataset["date"] = pd.to_datetime(dataset["date"], dayfirst=True)
dataset['quarter'] = dataset['date'].dt.to_period('Q')
dataset["quarter"] = dataset["quarter"].astype(str)
print(dataset[["date", "quarter"]].head())

dataset.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)

dataset = pd.read_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"]
)


print(dataset.head())

        date quarter
0 2024-12-31  2024Q4
1 2025-03-31  2025Q1
2 2025-06-30  2025Q2
3 2025-09-30  2025Q3
4 2025-12-31  2025Q4
  company        date                                       balancesheet  \
0    SNAP  2024-12-31  {'Other Current Liabilities': 140630000.0, 'Cu...   
1    SNAP  2025-03-31  {'Treasury Shares Number': 45980000.0, 'Ordina...   
2    SNAP  2025-06-30  {'Treasury Shares Number': 45577000.0, 'Ordina...   
3    SNAP  2025-09-30  {'Treasury Shares Number': 45157000.0, 'Ordina...   
4    SNAP  2025-12-31  {'Treasury Shares Number': 44670000.0, 'Ordina...   

                                            cashflow  \
0  {'Proceeds From Stock Option Exercised': 0.0, ...   
1  {'Free Cash Flow': 114396000.0, 'Repurchase Of...   
2  {'Free Cash Flow': 23793000.0, 'Repurchase Of ...   
3  {'Free Cash Flow': 93444000.0, 'Repurchase Of ...   
4  {'Free Cash Flow': 205556000.0, 'Repurchase Of...   

                                          financials quarter  
0                

C:\Users\phuon\AppData\Local\Temp\ipykernel_8908\3442495448.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dataset["date"] = pd.to_datetime(dataset["date"], dayfirst=True)


        date quarter
0 2024-09-30  2024Q3
1 2024-12-31  2024Q4
2 2025-03-31  2025Q1
3 2025-06-30  2025Q2
4 2025-09-30  2025Q3
